In [1]:
import serial
import time
from Pump import AL1000

ser = serial.Serial("COM2", 9600, timeout=1)
ser.write(b"@00SAF0\r")  # SAF0 disables Safe Mode
print(ser.read(64))


b'\x0200S?\x03'


In [3]:
ser.write(b"ID\r")
print(ser.read(64))



b'\x0200S?\x03'


In [3]:
pump.identify()

'No response'

In [4]:
ser.write(b"\r")
print(ser.read(64))


b''


In [1]:
import serial
import time

# Open the serial port (update COM port if needed)
ser = serial.Serial("COM2", 9600, timeout=1)

# Step 1: Disable Safe Mode
ser.write(b"SAF0\r")
time.sleep(0.5)  # give pump time to process

# Step 2: Query pump ID
ser.write(b"ID\r")
time.sleep(0.5)
response_id = ser.read(64)
print("ID response:", response_id)

# Step 3: Query syringe diameter
ser.write(b"DIA\r")
time.sleep(0.5)
response_dia = ser.read(64)
print("DIA response:", response_dia)

# Close the serial port when done
ser.close()


ID response: b'\x0200A?R\x03\x0200S?\x03'
DIA response: b'\x0200S32.30\x03'


In [4]:
import serial
import time

ser = serial.Serial("COM2", 9600, timeout=1)  # reopen port

# Stop any running program and clear dispensed volume
ser.write(b"STP\r")
time.sleep(0.2)
ser.write(b"CLD\r")
time.sleep(0.2)

# Read any responses
print(ser.read(64))
print(ser.read(64))



b'\x0200S?NA\x03\x0200S?\x03'
b''


In [5]:
ser.close()

In [6]:
from Pump import AL1000
import serial
import time

ser = serial.Serial("COM2", 9600, timeout=1)
pump = AL1000(ser)

pump.connect()        # should print "Device is connected"

# Set syringe diameter
pump.set_diameter(32.3)

# Set flow rate and direction
pump.set_rate(10)     # 10 uL/min
pump.set_direction("INF")  # INF for infuse

# Start the pump
pump.start()
time.sleep(2)         # run for 2 seconds
pump.stop()

# Check volume dispensed (if supported)
pump.command("DIS")   # optional

ser.close()


Device is connected
Sent: @00DIA32.3 | Reply: 00S?
Sent: @00RAT10 | Reply: 00S?
Sent: @00DIRINF | Reply: 00S?
Sent: @00STT | Reply: 00S?
Sent: @00STP | Reply: 00S?
Sent: @00DIS | Reply: 00S?


In [7]:
from Pump import AL1000
import serial
import time

# -------------------------
# 1️⃣ Open serial port
# -------------------------
ser = serial.Serial("COM2", 9600, timeout=1)

# -------------------------
# 2️⃣ Instantiate pump
# -------------------------
pump = AL1000(ser)
pump.connect()  # prints "Device is connected"

# -------------------------
# 3️⃣ Configure pump
# -------------------------
print("Setting syringe diameter...")
print(pump.set_diameter(32.3))   # mm

print("Setting flow rate...")
print(pump.set_rate(10))         # uL/min

print("Setting direction...")
print(pump.set_direction("INF")) # 'INF' or 'WDR'

print("Setting volume to dispense...")
print(pump.set_volume(5))        # uL

# -------------------------
# 4️⃣ Start the pump
# -------------------------
print("Starting pump...")
print(pump.start())
time.sleep(2)  # let it run for 2 seconds

# -------------------------
# 5️⃣ Stop the pump
# -------------------------
print("Stopping pump...")
print(pump.stop())

# -------------------------
# 6️⃣ Query dispensed volume
# -------------------------
print("Querying dispensed volume...")
print(pump.command("DIS"))

# -------------------------
# 7️⃣ Close serial port
# -------------------------
ser.close()
print("Serial port closed.")


Device is connected
Setting syringe diameter...
Sent: @00DIA32.3 | Reply: 00S?
00S?
Setting flow rate...
Sent: @00RAT10 | Reply: 00S?
00S?
Setting direction...
Sent: @00DIRINF | Reply: 00S?
00S?
Setting volume to dispense...
Sent: @00VOL5 | Reply: 00S?
00S?
Starting pump...
Sent: @00STT | Reply: 00S?
00S?
Stopping pump...
Sent: @00STP | Reply: 00S?
00S?
Querying dispensed volume...
Sent: @00DIS | Reply: 00S?
00S?
Serial port closed.


In [8]:
import serial
import time

ser = serial.Serial("COM2", 9600, timeout=1)

# Disable Safe Mode first
ser.write(b"@00SAF0\r")
time.sleep(0.5)
print("SAF0 response:", ser.read(64))

# Identify pump
ser.write(b"@00ID\r")
time.sleep(0.5)
print("ID response:", ser.read(64))

# Set diameter
ser.write(b"@00DIA32.3\r")
time.sleep(0.5)
print("DIA response:", ser.read(64))

ser.close()


SAF0 response: b'\x0200S?\x03'
ID response: b'\x0200S?\x03'
DIA response: b'\x0200S?\x03'


In [1]:
import serial
import time

# ----------------------------
# Setup serial connection
# ----------------------------
ser = serial.Serial(
    port="COM2",      # change if your COM is different
    baudrate=9600,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_NONE,
    stopbits=serial.STOPBITS_ONE,
    timeout=1
)

# Ensure port is open
if ser.is_open:
    print("Serial port opened successfully.")
else:
    ser.open()
    print("Serial port was closed, now opened.")

# ----------------------------
# Optional: Reset pump
# ----------------------------
ser.write(b"*RESET\r")
time.sleep(0.5)
response = ser.read(64)
print("RESET response:", response)

# ----------------------------
# Query firmware version
# ----------------------------
ser.write(b"*VER\r")
time.sleep(0.5)
response = ser.read(64)
print("Firmware response:", response)

# ----------------------------
# Close serial
# ----------------------------
ser.close()
print("Serial port closed.")


Serial port opened successfully.
RESET response: b'\x0200S\x03'
Firmware response: b'\x0200SNE1000V3.90\x03'
Serial port closed.


In [3]:
import serial
import time

# ---- Configuration ----
COM_PORT = "COM2"
BAUDRATE = 9600
ADDRESS = "@00"   # Pump address
TIMEOUT = 1

# ---- Helper function ----
def send_command(ser, cmd):
    full_cmd = f"{ADDRESS}{cmd}\r"
    ser.write(full_cmd.encode('ascii'))
    time.sleep(0.2)
    response = ser.read(64)
    print(f"Sent: {full_cmd.strip()} | Reply: {response}")
    return response

# ---- Open serial port ----
ser = serial.Serial(COM_PORT, BAUDRATE, timeout=TIMEOUT)
print("Serial port opened.")

# ---- 1. Query current syringe diameter ----
send_command(ser, "*DIA")

# ---- 2. Set a new diameter (e.g., 32.3 mm) ----
send_command(ser, "*DIA32.3")

# ---- 3. Query again to confirm ----
send_command(ser, "*DIA")

# ---- Close port ----
ser.close()
print("Serial port closed.")


Serial port opened.
Sent: @00*DIA | Reply: b'\x0200S32.30\x03'
Sent: @00*DIA32.3 | Reply: b'\x0200S\x03'
Sent: @00*DIA | Reply: b'\x0200S32.30\x03'
Serial port closed.
